In [13]:
# DATA CLEANING SETUP AND DATA LOADING

# Objectives:
# 1. Import the required libraries.
# 2. Identify the project directory.
# 3. Load the original outpatient appointment dataset.
# 4. Preserve an unchanged copy of the original data.
# 5. Create a separate working dataset for cleaning.
# 6. Validate the original dataset dimensions.

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Configure pandas display settings for easier inspection.

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

print("CELL 1: DATA CLEANING SETUP AND DATA LOADING")

# Identify the project root.

cwd = Path.cwd()

if (cwd / "data").is_dir():
    PROJECT_ROOT = cwd

elif (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Project directory not found. Open the notebook "
        "from the project root or notebooks folder."
    )

# Define the location of the original CSV dataset.

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Outpatient_appointments_synthetic.csv"
)

# Define the directory for cleaning reports.
# Create the directory if it does not already exist.

CLEANING_REPORT_DIR = (
    PROJECT_ROOT / "reports" / "cleaning"
)

CLEANING_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Confirm that the original dataset exists.

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

# Load the original dataset into a pandas DataFrame.

df = pd.read_csv(DATA_PATH)

# Preserve the original dataset as an independent copy.
# All subsequent cleaning operations will use df_clean.

df_raw = df.copy(deep=True)
df_clean = df_raw.copy(deep=True)

# Display the project and dataset information.

print("\nPROJECT INFORMATION")

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Cleaning report directory:", CLEANING_REPORT_DIR)

print("\nORIGINAL DATASET")

print(f"Number of rows: {len(df_raw):,}")
print(f"Number of columns: {df_raw.shape[1]}")

print("\nWORKING DATASET")

print(f"Number of rows: {len(df_clean):,}")
print(f"Number of columns: {df_clean.shape[1]}")

# Confirm that the original and working copies match.

assert df_raw.equals(df_clean), (
    "The original and working datasets do not match."
)

# Validate the expected original dataset dimensions.

assert df_raw.shape == (40180, 20), (
    "Dataset dimensions differ from the original profiling results."
)

# Display a sample of the original dataset.

print("\nFIRST FIVE RECORDS")

display(df_clean.head())

# Final validation.
# No transformations or row exclusions are applied here.

print("\nCELL 1 COMPLETED SUCCESSFULLY")
print("Original dataset preserved.")
print("Working dataset created.")
print("No cleaning transformations applied.")

CELL 1: DATA CLEANING SETUP AND DATA LOADING

PROJECT INFORMATION
Project root: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction
Dataset path: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\data\raw\Outpatient_appointments_synthetic.csv
Cleaning report directory: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning

ORIGINAL DATASET
Number of rows: 40,180
Number of columns: 20

WORKING DATASET
Number of rows: 40,180
Number of columns: 20

FIRST FIVE RECORDS


,appointment_id,patient_id,age,sex,imd_decile,ethnic_group,distance_km,interpreter_required,long_term_conditions,specialty,appointment_type,consultation_mode,referral_source,appointment_datetime,booked_datetime,rescheduled_count,sms_reminder_sent,previous_dnas_12m,attendance_status,clinic_checkin_time
0,OPA2004425,P113753,70,M,2.0,Not Stated,5.7,N,1,Respiratory Medicine,Follow-up,Video,Consultant,02/09/2025 11:30,07/06/2025 12:59,1,Y,1,Attended,11:16
1,OPA2000901,P119960,58,Female,2.0,White,4.7,N,0,Cardiology,Follow-up,Face-to-face,Consultant,12/11/2025 10:30,13/08/2025 12:33,0,Y,0,Attended,10:13
2,OPA2014471,P122474,76,Female,9.0,White,18.1,N,1,Diabetes and Endocrinology,New,Face-to-face,Other,17/11/2025 13:30,03/10/2025 14:08,0,Y,0,Cancelled by hospital,NaN
3,OPA2002727,P113052,69,Male,5.0,White,13.1,N,3,Trauma and Orthopaedics,Follow-up,Face-to-face,Consultant,14/07/2025 14:30,05/06/2025 16:40,1,Y,0,Attended,14:22
4,OPA2007989,P120866,72,Female,9.0,White,4.1,N,1,Opthalmology,Follow-up,Face-to-face,Consultant,13/10/2025 10:00,27/08/2025 17:14,0,Y,0,Attended,09:38



CELL 1 COMPLETED SUCCESSFULLY
Original dataset preserved.
Working dataset created.
No cleaning transformations applied.


In [14]:
# IDENTIFY AND REMOVE EXACT DUPLICATE RECORDS
# NHS DORSET - OUTPATIENT DNA PREDICTION

# Objectives:
# 1. Identify exact duplicate appointment records.
# 2. Calculate the number and percentage of duplicates.
# 3. Check repeated appointment identifiers.
# 4. Save duplicate records for auditing.
# 5. Remove exact duplicates from the working dataset.
# 6. Validate the cleaning results.

print("EXACT DUPLICATE IDENTIFICATION AND REMOVAL")

# 1. Record the original working dataset size.

rows_before = len(df_clean)

print("\n1. DATASET BEFORE DUPLICATE REMOVAL")
print(f"Total appointments: {rows_before:,}")

# 2. Identify exact duplicates across all columns.


duplicate_mask = df_clean.duplicated(keep="first")

duplicate_count = int(duplicate_mask.sum())

duplicate_percentage = (
    duplicate_count / rows_before * 100
    if rows_before > 0 else 0
)

print("\n2. EXACT DUPLICATE CHECK")
print(f"Exact duplicates identified: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

# 3. Check whether appointment identifiers are repeated.
# Repeated identifiers are investigated separately because
# two records with the same ID are not necessarily identical.

duplicate_ids_before = int(
    df_clean["appointment_id"].duplicated().sum()
)

print("\n3. APPOINTMENT IDENTIFIER CHECK")
print(f"Repeated appointment IDs before cleaning: {duplicate_ids_before:,}")

# 4. Save duplicate records for auditing.
# This report preserves the removed records and their values.

duplicate_records = df_clean.loc[duplicate_mask].copy()

duplicate_report_path = (
    CLEANING_REPORT_DIR / "exact_duplicate_records.csv"
)

duplicate_records.to_csv(
    duplicate_report_path,
    index=False
)

print("\n4. DUPLICATE AUDIT REPORT")
print(f"Audit report saved to: {duplicate_report_path}")

# 5. Remove exact duplicates from the working dataset only.
# The original dataset df_raw remains unchanged.

df_clean = (
    df_clean
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

# 6. Calculate the number of records remaining.

rows_after = len(df_clean)

records_removed = rows_before - rows_after

duplicates_remaining = int(
    df_clean.duplicated().sum()
)

duplicate_ids_after = int(
    df_clean["appointment_id"].duplicated().sum()
)

print("\n5. DATASET AFTER DUPLICATE REMOVAL")

print(f"Records before cleaning: {rows_before:,}")
print(f"Records removed: {records_removed:,}")
print(f"Records remaining: {rows_after:,}")
print(f"Exact duplicates remaining: {duplicates_remaining:,}")
print(f"Repeated appointment IDs remaining: {duplicate_ids_after:,}")

# 7. Validate the cleaning operation.
# The number removed must equal the number identified.

assert records_removed == duplicate_count, (
    "Duplicate removal count does not match the audit."
)

assert duplicates_remaining == 0, (
    "Exact duplicate records remain."
)

assert duplicate_ids_after == 0, (
    "Repeated appointment IDs remain. Investigate before continuing."
)

assert len(df_raw) == 40180, (
    "The original dataset row count has changed."
)

# 8. Document the cleaning decision.

cleaning_decision = {
    "Step": "Exact duplicate removal",
    "Issue": "Repeated identical appointment records",
    "Action": "Retain first occurrence and remove exact duplicates",
    "Records_Affected": records_removed,
    "Reason": (
        "Prevent repeated appointments from distorting "
        "attendance statistics and model training."
    )
}

print("\n6. CLEANING DECISION")

for key, value in cleaning_decision.items():
    print(f"{key}: {value}")

# 9. Final confirmation.

print("\nCELL 2 COMPLETED SUCCESSFULLY")
print("Original dataset preserved.")
print("Duplicate audit report saved.")
print("Working dataset updated.")

EXACT DUPLICATE IDENTIFICATION AND REMOVAL

1. DATASET BEFORE DUPLICATE REMOVAL
Total appointments: 40,180

2. EXACT DUPLICATE CHECK
Exact duplicates identified: 180
Duplicate percentage: 0.45%

3. APPOINTMENT IDENTIFIER CHECK
Repeated appointment IDs before cleaning: 180

4. DUPLICATE AUDIT REPORT
Audit report saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\exact_duplicate_records.csv

5. DATASET AFTER DUPLICATE REMOVAL
Records before cleaning: 40,180
Records removed: 180
Records remaining: 40,000
Exact duplicates remaining: 0
Repeated appointment IDs remaining: 0

6. CLEANING DECISION
Step: Exact duplicate removal
Issue: Repeated identical appointment records
Action: Retain first occurrence and remove exact duplicates
Records_Affected: 180
Reason: Prevent repeated appointments from distorting attendance statistics and model training.

CELL 2 COMPLETED SUCCESSFULLY
Original dataset preserved.
Duplicate audit report saved.
Working dataset update

In [15]:
# INVESTIGATE INVALID AND SUSPICIOUS AGES
# NHS DORSET - OUTPATIENT DNA PREDICTION

# Objectives:
# 1. Examine the age distribution.
# 2. Identify negative ages and ages above 120.
# 3. Examine affected appointment records.
# 4. Save an audit report.
# 5. Preserve all records until the cleaning decision is applied.

print("INVESTIGATION OF INVALID AND SUSPICIOUS AGES")

# 1. Confirm the current dataset size.

rows_before = len(df_clean)

print("\n1. DATASET INFORMATION")
print(f"Total appointments: {rows_before:,}")

# 2. Convert ages to numeric for investigation only.
# The working dataset is not modified in this cell.

age_numeric = pd.to_numeric(
    df_clean["age"],
    errors="coerce"
)

print("\n2. AGE DISTRIBUTION")
display(age_numeric.describe())

# 3. Flag ages outside the accepted range.
# Ages below 0 are invalid.
# Ages above 120 are considered suspicious.
# Missing values are reported separately.

invalid_age_mask = (
    (age_numeric < 0) |
    (age_numeric > 120)
)

invalid_age_records = df_clean.loc[
    invalid_age_mask
].copy()

invalid_age_count = int(invalid_age_mask.sum())

print("\n3. INVALID AGE CHECK")
print(f"Invalid or suspicious ages: {invalid_age_count:,}")
print(f"Percentage affected: {invalid_age_mask.mean() * 100:.2f}%")
print(f"Missing or nonnumeric ages: {age_numeric.isna().sum():,}")

# 4. Examine the flagged age values.

print("\n4. INVALID AGE VALUES")

display(
    age_numeric.loc[invalid_age_mask]
    .value_counts()
    .sort_index()
    .rename_axis("Age")
    .reset_index(name="Appointments")
)

# 5. Examine attendance outcomes for affected records.

print("\n5. ATTENDANCE STATUS OF AFFECTED RECORDS")

display(
    invalid_age_records["attendance_status"]
    .value_counts(dropna=False)
    .rename_axis("Attendance_Status")
    .reset_index(name="Appointments")
)

# 6. Display a sample for investigation.

print("\n6. SAMPLE FLAGGED APPOINTMENTS")

display(
    invalid_age_records[
        [
            "appointment_id",
            "patient_id",
            "age",
            "attendance_status"
        ]
    ].head(15)
)

# 7. Save the affected records for audit purposes.

invalid_age_report_path = (
    CLEANING_REPORT_DIR / "suspicious_age_records.csv"
)

invalid_age_records.to_csv(
    invalid_age_report_path,
    index=False
)

print("\n7. AUDIT REPORT")
print(f"Report saved to: {invalid_age_report_path}")

# 8. Confirm that no records were modified or removed.

assert len(df_clean) == rows_before == 40000, (
    "Unexpected change in working dataset size."
)

assert len(df_raw) == 40180, (
    "Original dataset has changed."
)

assert df_clean["appointment_id"].is_unique, (
    "Duplicate appointment IDs detected."
)

print("\nCELL 3 COMPLETED SUCCESSFULLY")
print("Suspicious ages identified for review.")
print("No records removed or modified.")
print("Original dataset preserved.")

INVESTIGATION OF INVALID AND SUSPICIOUS AGES

1. DATASET INFORMATION
Total appointments: 40,000

2. AGE DISTRIBUTION


count    40000.000000
mean        52.484375
std         28.734969
min         -1.000000
25%         31.000000
50%         58.000000
75%         74.000000
max        999.000000
Name: age, dtype: float64


3. INVALID AGE CHECK
Invalid or suspicious ages: 54
Percentage affected: 0.14%
Missing or nonnumeric ages: 0

4. INVALID AGE VALUES


,Age,Appointments
0,-1,35
1,121,4
2,150,7
3,999,8



5. ATTENDANCE STATUS OF AFFECTED RECORDS


,Attendance_Status,Appointments
0,Attended,46
1,Cancelled by patient,3
2,DNA,3
3,Cancelled by hospital,2



6. SAMPLE FLAGGED APPOINTMENTS


,appointment_id,patient_id,age,attendance_status
112,OPA2029740,P108262,-1,Attended
450,OPA2025852,P101313,-1,Attended
710,OPA2011607,P105638,-1,Attended
736,OPA2033313,P109731,999,Attended
1068,OPA2022893,P102698,-1,Attended
1394,OPA2021020,P103686,-1,Attended
1705,OPA2005193,P102854,-1,Cancelled by patient
1945,OPA2007876,P107902,999,Attended
2023,OPA2028687,P107032,-1,Attended
2412,OPA2025705,P118674,999,DNA



7. AUDIT REPORT
Report saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\suspicious_age_records.csv

CELL 3 COMPLETED SUCCESSFULLY
Suspicious ages identified for review.
No records removed or modified.
Original dataset preserved.


In [16]:
#  HANDLE INVALID AGE VALUES


# Objectives:
# 1. Convert age to a numeric data type.
# 2. Replace invalid ages with missing values.
# 3. Preserve all appointment records.
# 4. Validate the cleaning operation.
# 5. Document the cleaning decision.

print("CELL 4: HANDLE INVALID AGE VALUES")

# 1. Record the dataset size before cleaning.

rows_before = len(df_clean)

print("\n1. DATASET BEFORE AGE CLEANING")
print(f"Total appointments: {rows_before:,}")

# 2. Convert age to numeric.
# Values that cannot be converted become NaN.

df_clean["age"] = pd.to_numeric(
    df_clean["age"],
    errors="coerce"
)

# 3. Identify invalid age values.
# Negative ages and ages above 120 are treated as invalid.

invalid_age_mask = (
    (df_clean["age"] < 0) |
    (df_clean["age"] > 120)
)

invalid_age_count = int(invalid_age_mask.sum())

print("\n2. INVALID AGES IDENTIFIED")
print(f"Invalid ages: {invalid_age_count:,}")

# 4. Replace invalid ages with missing values.


df_clean.loc[invalid_age_mask, "age"] = np.nan

# 5. Validate the cleaned age values.

remaining_invalid = int(
    (
        (df_clean["age"] < 0) |
        (df_clean["age"] > 120)
    ).sum()
)

missing_age_count = int(
    df_clean["age"].isna().sum()
)

print("\n3. AGE CLEANING RESULTS")

print(f"Invalid ages corrected: {invalid_age_count:,}")
print(f"Invalid ages remaining: {remaining_invalid:,}")
print(f"Missing age values: {missing_age_count:,}")

print("\n4. CLEANED AGE DISTRIBUTION")

display(df_clean["age"].describe())

# 6. Confirm that no appointment records were removed.

rows_after = len(df_clean)

assert rows_after == rows_before == 40000, (
    "Unexpected change in appointment count."
)

assert remaining_invalid == 0, (
    "Invalid ages remain after cleaning."
)

assert df_clean["appointment_id"].is_unique, (
    "Duplicate appointment IDs detected."
)

assert len(df_raw) == 40180, (
    "Original dataset row count has changed."
)

# 7. Document the cleaning decision.

print("\n5. CLEANING DECISION")

print("Issue: Invalid patient age values")
print("Action: Replace invalid ages with NaN")
print(f"Values affected: {invalid_age_count:,}")
print("Appointment records removed: 0")

print(
    "Reason: Invalid age values should not influence "
    "model predictions, but otherwise valid appointments "
    "should be retained."
)

print(
    "Missing age values will be handled during "
    "model preprocessing using training data only."
)

print("\nCELL 4 COMPLETED SUCCESSFULLY")
print("Invalid age values replaced with NaN.")
print("All appointment records retained.")
print("Original dataset preserved.")

CELL 4: HANDLE INVALID AGE VALUES

1. DATASET BEFORE AGE CLEANING
Total appointments: 40,000

2. INVALID AGES IDENTIFIED
Invalid ages: 54

3. AGE CLEANING RESULTS
Invalid ages corrected: 54
Invalid ages remaining: 0
Missing age values: 54

4. CLEANED AGE DISTRIBUTION


count    39946.000000
mean        52.317729
std         25.351901
min          0.000000
25%         31.000000
50%         58.000000
75%         74.000000
max        118.000000
Name: age, dtype: float64


5. CLEANING DECISION
Issue: Invalid patient age values
Action: Replace invalid ages with NaN
Values affected: 54
Appointment records removed: 0
Reason: Invalid age values should not influence model predictions, but otherwise valid appointments should be retained.
Missing age values will be handled during model preprocessing using training data only.

CELL 4 COMPLETED SUCCESSFULLY
Invalid age values replaced with NaN.
All appointment records retained.
Original dataset preserved.


In [17]:
# INVESTIGATE INCONSISTENT CATEGORICAL VALUES


# Objectives:
# 1. Identify categorical columns.
# 2. Examine unique values and their frequencies.
# 3. Investigate inconsistent sex categories.
# 4. Investigate specialty spelling inconsistencies.
# 5. Check missing categorical values.
# 6. Save a categorical profiling report.
# 7. Preserve all records without applying transformations.

print("CELL 5: CATEGORICAL DATA QUALITY INVESTIGATION")

# 1. Confirm dataset size before investigation.

rows_before = len(df_clean)

print("\n1. DATASET INFORMATION")
print(f"Total appointments: {rows_before:,}")
print(f"Total columns: {df_clean.shape[1]}")

# 2. Identify categorical columns.
# Include object, string, and category data types.


categorical_columns = df_clean.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("\n2. CATEGORICAL COLUMNS IDENTIFIED")

for column in categorical_columns:
    print(column)

print(f"\nTotal categorical columns: {len(categorical_columns)}")

# 3. Create a categorical data quality summary.
# Count distinct non-missing values and missing values
# for each categorical column.

categorical_summary = []

for column in categorical_columns:

    categorical_summary.append({
        "column": column,
        "unique_values": df_clean[column].nunique(dropna=True),
        "missing_values": int(df_clean[column].isna().sum()),
        "missing_percentage": round(
            df_clean[column].isna().mean() * 100,
            2
        )
    })

categorical_summary_df = pd.DataFrame(
    categorical_summary
)

print("\n3. CATEGORICAL DATA QUALITY SUMMARY")

display(categorical_summary_df)

# 4. Investigate sex categories.
# Different representations such as M, Male, F, and Female
# may refer to the same categories.


print("\n4. SEX CATEGORY INVESTIGATION")

if "sex" in df_clean.columns:

    print("Original sex values and frequencies:")

    display(
        df_clean["sex"]
        .value_counts(dropna=False)
        .rename_axis("Sex")
        .reset_index(name="Appointments")
    )

    print(
        "Distinct non-missing sex values:",
        df_clean["sex"].nunique(dropna=True)
    )

else:
    print("Column 'sex' not found.")

# 5. Investigate specialty categories.
# Look for inconsistent spellings and categories
# that may represent the same clinical specialty.

print("\n5. SPECIALTY CATEGORY INVESTIGATION")

if "specialty" in df_clean.columns:

    display(
        df_clean["specialty"]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("Specialty")
        .reset_index(name="Appointments")
    )

    print(
        "Distinct non-missing specialties:",
        df_clean["specialty"].nunique(dropna=True)
    )

else:
    print("Column 'specialty' not found.")

# 6. Investigate the remaining categorical columns.

print("\n6. OTHER CATEGORICAL COLUMNS")

for column in categorical_columns:

    if column not in ["sex", "specialty"]:

        unique_count = df_clean[column].nunique(
            dropna=True
        )

        print(f"\nCOLUMN: {column}")
        print(f"Unique values: {unique_count:,}")

        if unique_count <= 30:

            display(
                df_clean[column]
                .value_counts(dropna=False)
                .rename_axis(column)
                .reset_index(name="Appointments")
            )

        else:
            print(
                "More than 30 unique values. "
                "Full frequency listing skipped."
            )

# 7. Save the categorical profiling report.
# This report documents the original categorical values
# before standardization.

categorical_report_path = (
    CLEANING_REPORT_DIR
    / "categorical_quality_summary.csv"
)

categorical_summary_df.to_csv(
    categorical_report_path,
    index=False
)

print("\n7. CATEGORICAL PROFILING REPORT")
print(f"Report saved to: {categorical_report_path}")

# 8. Confirm that the investigation has not removed
# any appointments or changed the original dataset.

rows_after = len(df_clean)

assert rows_before == rows_after == 40000, (
    "Unexpected change in appointment count."
)

assert len(df_raw) == 40180, (
    "Original dataset row count has changed."
)

assert df_clean["appointment_id"].is_unique, (
    "Duplicate appointment IDs detected."
)

# 9. Final confirmation.

print("\nCELL 5 COMPLETED SUCCESSFULLY")
print(f"Appointments investigated: {rows_after:,}")
print(f"Categorical columns investigated: {len(categorical_columns)}")
print("No categorical values modified.")
print("No appointment records removed.")
print("Original dataset preserved.")

CELL 5: CATEGORICAL DATA QUALITY INVESTIGATION

1. DATASET INFORMATION
Total appointments: 40,000
Total columns: 20

2. CATEGORICAL COLUMNS IDENTIFIED
appointment_id
patient_id
sex
ethnic_group
interpreter_required
specialty
appointment_type
consultation_mode
referral_source
appointment_datetime
booked_datetime
sms_reminder_sent
attendance_status
clinic_checkin_time

Total categorical columns: 14

3. CATEGORICAL DATA QUALITY SUMMARY


,column,unique_values,missing_values,missing_percentage
0,appointment_id,40000,0,0.00
1,patient_id,15458,0,0.00
2,sex,7,0,0.00
3,ethnic_group,6,0,0.00
4,interpreter_required,2,0,0.00
5,specialty,9,0,0.00
6,appointment_type,2,0,0.00
7,consultation_mode,3,0,0.00
8,referral_source,4,0,0.00
9,appointment_datetime,4427,0,0.00



4. SEX CATEGORY INVESTIGATION
Original sex values and frequencies:


,Sex,Appointments
0,Female,20419
1,Male,18181
2,F,364
3,M,336
4,female,307
5,male,293
6,Unknown,100


Distinct non-missing sex values: 7

5. SPECIALTY CATEGORY INVESTIGATION


,Specialty,Appointments
0,Cardiology,4062
1,Dermatology,4351
2,Diabetes and Endocrinology,3311
3,Gastroentterology,3690
4,Opthalmology,4679
5,Paediatrics,3378
6,Physiotherapy,4473
7,Respiratory Medicine,3250
8,Trauma and Orthopaedics,8806


... (11 outputs left out)

,sms_reminder_sent,Appointments
0,Y,30431
1,N,9569



COLUMN: attendance_status
Unique values: 4


,attendance_status,Appointments
0,Attended,33048
1,DNA,3316
2,Cancelled by patient,2421
3,Cancelled by hospital,1215



COLUMN: clinic_checkin_time
Unique values: 520
More than 30 unique values. Full frequency listing skipped.

7. CATEGORICAL PROFILING REPORT
Report saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\categorical_quality_summary.csv

CELL 5 COMPLETED SUCCESSFULLY
Appointments investigated: 40,000
Categorical columns investigated: 14
No categorical values modified.
No appointment records removed.
Original dataset preserved.


In [18]:
# STANDARDIZE INCONSISTENT CATEGORICAL VALUES

# Objectives:
# 1. Standardize inconsistent sex categories.
# 2. Correct known specialty spelling errors.
# 3. Preserve unknown sex values.
# 4. Validate categorical transformations.
# 5. Retain all appointment records.
# 6. Save a summary of the cleaning decisions.

print("CATEGORICAL VALUE STANDARDIZATION")

# 1. Record the dataset size before cleaning.

rows_before = len(df_clean)

print("\n1. DATASET BEFORE CATEGORICAL CLEANING")
print(f"Total appointments: {rows_before:,}")

# 2. Preserve original values for comparison.

sex_before = df_clean["sex"].copy()
specialty_before = df_clean["specialty"].copy()

# 3. Standardize sex values.
# Convert values to strings, remove surrounding spaces,
# and normalize capitalization before applying mappings.
# Unknown values are retained .

sex_normalized = (
    df_clean["sex"]
    .astype("string")
    .str.strip()
    .str.lower()
)

sex_mapping = {
    "f": "Female",
    "female": "Female",
    "m": "Male",
    "male": "Male",
    "unknown": "Unknown"
}

df_clean["sex"] = sex_normalized.map(sex_mapping)

# 4. Validate the standardized sex categories.

print("\n2. STANDARDIZED SEX CATEGORIES")

display(
    df_clean["sex"]
    .value_counts(dropna=False)
    .rename_axis("Sex")
    .reset_index(name="Appointments")
)

unexpected_sex_values = set(
    df_clean["sex"].dropna().unique()
) - {"Female", "Male", "Unknown"}

assert not unexpected_sex_values, (
    f"Unexpected sex categories: {unexpected_sex_values}"
)

# 5. Standardize specialty names.
# Remove surrounding whitespace before correcting
# the two spelling inconsistencies identified in Cell 5.

df_clean["specialty"] = (
    df_clean["specialty"]
    .astype("string")
    .str.strip()
)

specialty_mapping = {
    "Gastroentterology": "Gastroenterology",
    "Opthalmology": "Ophthalmology"
}

df_clean["specialty"] = (
    df_clean["specialty"]
    .replace(specialty_mapping)
)

# 6. Validate the standardized specialty categories.

print("\n3. STANDARDIZED SPECIALTY CATEGORIES")

display(
    df_clean["specialty"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Specialty")
    .reset_index(name="Appointments")
)

assert "Gastroentterology" not in set(
    df_clean["specialty"].dropna()
), "Misspelled Gastroenterology category remains."

assert "Opthalmology" not in set(
    df_clean["specialty"].dropna()
), "Misspelled Ophthalmology category remains."

# 7. Calculate the number of values changed.

sex_changes = int(
    (
        sex_before.astype("string")
        != df_clean["sex"]
    ).fillna(False).sum()
)

specialty_changes = int(
    (
        specialty_before.astype("string")
        != df_clean["specialty"]
    ).fillna(False).sum()
)

print("\n4. CATEGORICAL CLEANING SUMMARY")

print(f"Sex values standardized: {sex_changes:,}")
print(f"Specialty values corrected: {specialty_changes:,}")

print(
    "Final sex categories:",
    df_clean["sex"].nunique(dropna=True)
)

print(
    "Final specialty categories:",
    df_clean["specialty"].nunique(dropna=True)
)

# 8. Confirming that the transformations did not introduce
# missing values or remove appointment records.

assert df_clean["sex"].isna().sum() == sex_before.isna().sum(), (
    "Unexpected missing sex values introduced."
)

assert df_clean["specialty"].isna().sum() == specialty_before.isna().sum(), (
    "Unexpected missing specialty values introduced."
)

assert len(df_clean) == rows_before == 40000, (
    "Unexpected change in appointment count."
)

assert df_clean["appointment_id"].is_unique, (
    "Duplicate appointment IDs detected."
)

assert len(df_raw) == 40180, (
    "Original dataset row count has changed."
)

# 9. Create a cleaning audit summary.

categorical_cleaning_summary = pd.DataFrame([
    {
        "column": "sex",
        "action": "Standardize sex categories",
        "values_changed": sex_changes,
        "reason": (
            "Different representations of the same sex "
            "category should be treated consistently."
        )
    },
    {
        "column": "specialty",
        "action": "Correct known specialty spelling errors",
        "values_changed": specialty_changes,
        "reason": (
            "Correct misspellings to avoid creating "
            "artificially separate specialty categories."
        )
    }
])

print("\n5. CLEANING AUDIT SUMMARY")

display(categorical_cleaning_summary)

# 10. Save the cleaning audit summary.

categorical_cleaning_report_path = (
    CLEANING_REPORT_DIR
    / "categorical_standardization_summary.csv"
)

categorical_cleaning_summary.to_csv(
    categorical_cleaning_report_path,
    index=False
)

print("\n6. AUDIT REPORT")
print(f"Report saved to: {categorical_cleaning_report_path}")

# 11. Final confirmation.

print("\nCELL 6 COMPLETED SUCCESSFULLY")
print("Sex categories standardized.")
print("Specialty spelling errors corrected.")
print("Unknown sex values retained.")
print("All 40,000 appointments preserved.")
print("Original dataset unchanged.")

CATEGORICAL VALUE STANDARDIZATION

1. DATASET BEFORE CATEGORICAL CLEANING
Total appointments: 40,000

2. STANDARDIZED SEX CATEGORIES


,Sex,Appointments
0,Female,21090
1,Male,18810
2,Unknown,100



3. STANDARDIZED SPECIALTY CATEGORIES


,Specialty,Appointments
0,Cardiology,4062
1,Dermatology,4351
2,Diabetes and Endocrinology,3311
3,Gastroenterology,3690
4,Ophthalmology,4679
5,Paediatrics,3378
6,Physiotherapy,4473
7,Respiratory Medicine,3250
8,Trauma and Orthopaedics,8806



4. CATEGORICAL CLEANING SUMMARY
Sex values standardized: 1,300
Specialty values corrected: 8,369
Final sex categories: 3
Final specialty categories: 9

5. CLEANING AUDIT SUMMARY


,column,action,values_changed,reason
0,sex,Standardize sex categories,1300,Different representations of the same sex cate...
1,specialty,Correct known specialty spelling errors,8369,Correct misspellings to avoid creating artific...



6. AUDIT REPORT
Report saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\categorical_standardization_summary.csv

CELL 6 COMPLETED SUCCESSFULLY
Sex categories standardized.
Specialty spelling errors corrected.
Unknown sex values retained.
All 40,000 appointments preserved.
Original dataset unchanged.


In [19]:
# INVESTIGATE MISSING VALUES


# Objectives:
# 1. Identify missing values across all columns.
# 2. Calculate missing-value percentages.
# 3. Examine missingness in important columns.
# 4. Compare missingness across attendance outcomes.
# 5. Identify potential target leakage.
# 6. Save a missing-value profiling report.
# 7. Preserve all appointment records.

print("CELL 7: MISSING VALUE INVESTIGATION")

# 1. Record the dataset dimensions.

rows_before = len(df_clean)

print("\n1. DATASET INFORMATION")
print(f"Total appointments: {rows_before:,}")
print(f"Total columns: {df_clean.shape[1]}")

# 2. Calculate missing values for every column.

missing_summary = pd.DataFrame({
    "column": df_clean.columns,
    "missing_count": df_clean.isna().sum().values,
    "missing_percentage": (
        df_clean.isna().mean().values * 100
    ).round(2)
})

missing_summary = (
    missing_summary
    .sort_values("missing_count", ascending=False)
    .reset_index(drop=True)
)

print("\n2. MISSING VALUE SUMMARY")
display(missing_summary)

# 3. Identify columns containing missing values.

columns_with_missing = missing_summary.loc[
    missing_summary["missing_count"] > 0,
    "column"
].tolist()

print("\n3. COLUMNS WITH MISSING VALUES")

if columns_with_missing:

    for column in columns_with_missing:

        count = int(df_clean[column].isna().sum())
        percentage = df_clean[column].isna().mean() * 100

        print(
            f"{column}: {count:,} missing "
            f"({percentage:.2f}%)"
        )

else:
    print("No missing values identified.")

# 4. Examine missingness in important columns.

important_columns = [
    "age",
    "imd_decile",
    "distance_km",
    "clinic_checkin_time"
]

print("\n4. IMPORTANT COLUMNS")

for column in important_columns:

    if column in df_clean.columns:

        missing_count = int(
            df_clean[column].isna().sum()
        )

        missing_pct = (
            df_clean[column].isna().mean() * 100
        )

        print(f"\nColumn: {column}")
        print(f"Missing records: {missing_count:,}")
        print(f"Missing percentage: {missing_pct:.2f}%")

# 5. Examine missingness by attendance status.
# This is a descriptive data quality investigation

print("\n5. MISSING VALUES BY ATTENDANCE STATUS")

missing_by_attendance = {}

for column in important_columns:

    if column in df_clean.columns:

        result = (
            df_clean
            .assign(
                is_missing=df_clean[column].isna()
            )
            .groupby("attendance_status")["is_missing"]
            .agg(
                missing_count="sum",
                total_appointments="count",
                missing_percentage="mean"
            )
        )

        result["missing_percentage"] = (
            result["missing_percentage"] * 100
        ).round(2)

        missing_by_attendance[column] = result

        print(f"\nCOLUMN: {column}")
        display(result)

# 6. Investigate clinic check-in time separately.

print("\n6. CLINIC CHECK-IN TIME INVESTIGATION")

if "clinic_checkin_time" in df_clean.columns:

    checkin_missing = (
        df_clean["clinic_checkin_time"].isna()
    )

    checkin_summary = pd.crosstab(
        df_clean["attendance_status"],
        checkin_missing
    )

    checkin_summary = checkin_summary.rename(
        columns={
            False: "Check-in recorded",
            True: "Check-in missing"
        }
    )

    display(checkin_summary)

    print("\nTARGET LEAKAGE WARNING")

    print(
        "clinic_checkin_time is not available "
        "before the appointment."
    )

    print(
        "Exclude both its value and its missingness "
        "indicator from predictive features."
    )

# 7. Count missing values per appointment.

missing_per_row = df_clean.isna().sum(axis=1)

print("\n7. MISSING VALUES PER APPOINTMENT")

display(
    missing_per_row
    .value_counts()
    .sort_index()
    .rename_axis("Missing_Fields")
    .reset_index(name="Appointments")
)

appointments_with_missing = int(
    (missing_per_row > 0).sum()
)

print(
    f"Appointments with at least one missing value: "
    f"{appointments_with_missing:,}"
)

# 8. Save the missing-value profiling report.

missing_report_path = (
    CLEANING_REPORT_DIR / "missing_value_summary.csv"
)

missing_summary.to_csv(
    missing_report_path,
    index=False
)

print("\n8. MISSING VALUE REPORT")
print(f"Report saved to: {missing_report_path}")

# 9. Validate that the investigation has not
# modified or removed appointment records.

assert len(df_clean) == rows_before == 40000, (
    "Unexpected change in appointment count."
)

assert len(df_raw) == 40180, (
    "Original dataset row count has changed."
)

assert df_clean["appointment_id"].is_unique, (
    "Duplicate appointment IDs detected."
)

# 10. Document the cleaning decision.

print("\n9. CLEANING DECISION")

print("Action: Retain records with missing values.")

print(
    "Reason: Missing values do not automatically "
    "make an appointment invalid."
)

print(
    "Predictor imputation will be fitted using "
    "training data only to prevent data leakage."
)

print(
    "clinic_checkin_time will not be used "
    "for pre-appointment prediction."
)

print("\nCELL 7 COMPLETED SUCCESSFULLY")
print("Missing values investigated.")
print("No missing values imputed.")
print("No columns removed.")
print("All 40,000 appointments retained.")
print("Original dataset preserved.")

CELL 7: MISSING VALUE INVESTIGATION

1. DATASET INFORMATION
Total appointments: 40,000
Total columns: 20

2. MISSING VALUE SUMMARY


,column,missing_count,missing_percentage
0,clinic_checkin_time,6952,17.38
1,imd_decile,1400,3.50
2,distance_km,400,1.00
3,age,54,0.14
4,sex,0,0.00
5,patient_id,0,0.00
6,appointment_id,0,0.00
7,ethnic_group,0,0.00
8,long_term_conditions,0,0.00
9,specialty,0,0.00



3. COLUMNS WITH MISSING VALUES
clinic_checkin_time: 6,952 missing (17.38%)
imd_decile: 1,400 missing (3.50%)
distance_km: 400 missing (1.00%)
age: 54 missing (0.14%)

4. IMPORTANT COLUMNS

Column: age
Missing records: 54
Missing percentage: 0.14%

Column: imd_decile
Missing records: 1,400
Missing percentage: 3.50%

Column: distance_km
Missing records: 400
Missing percentage: 1.00%

Column: clinic_checkin_time
Missing records: 6,952
Missing percentage: 17.38%

5. MISSING VALUES BY ATTENDANCE STATUS

COLUMN: age


,missing_count,total_appointments,missing_percentage
attendance_status,,,
Attended,46,33048,0.14
Cancelled by hospital,2,1215,0.16
Cancelled by patient,3,2421,0.12
DNA,3,3316,0.09



COLUMN: imd_decile


,missing_count,total_appointments,missing_percentage
attendance_status,,,
Attended,1162,33048,3.52
Cancelled by hospital,50,1215,4.12
Cancelled by patient,82,2421,3.39
DNA,106,3316,3.20


... (5 outputs left out)

clinic_checkin_time,Check-in recorded,Check-in missing
attendance_status,,
Attended,33048,0
Cancelled by hospital,0,1215
Cancelled by patient,0,2421
DNA,0,3316



TARGET LEAKAGE WARNING
clinic_checkin_time is not available before the appointment.
Exclude both its value and its missingness indicator from predictive features.

7. MISSING VALUES PER APPOINTMENT


,Missing_Fields,Appointments
0,0,31520
1,1,8157
2,2,320
3,3,3


Appointments with at least one missing value: 8,480

8. MISSING VALUE REPORT
Report saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\missing_value_summary.csv

9. CLEANING DECISION
Action: Retain records with missing values.
Reason: Missing values do not automatically make an appointment invalid.
Predictor imputation will be fitted using training data only to prevent data leakage.
clinic_checkin_time will not be used for pre-appointment prediction.

CELL 7 COMPLETED SUCCESSFULLY
Missing values investigated.
No missing values imputed.
No columns removed.
All 40,000 appointments retained.
Original dataset preserved.


In [20]:
#  DATETIME VALIDATION AND BOOKING LEAD-TIME ANALYSIS


# Objectives:
# 1. Validate appointment and booking datetime fields.
# 2. Calculate the original booking lead time in days.
# 3. Identify missing or invalid datetime values.
# 4. Identify appointments booked after their scheduled time.
# 5. Investigate same-day and short-notice bookings.
# 6. Apply the agreed 0–301-day booking lead-time range.
# 7. Define the dynamic prediction timing strategy.
# 8. Save invalid booking records for auditing.
# 9. Preserve all 40,000 appointments in the cleaned dataset.

print("DATETIME VALIDATION AND BOOKING LEAD-TIME ANALYSIS")

# 1. Confirm the dataset dimensions before processing.

rows_before = len(df_clean)

print("\n1. DATASET INFORMATION")
print(f"Total appointments: {rows_before:,}")
print(f"Total columns before processing: {df_clean.shape[1]}")

# 2. Convert datetime columns to pandas datetime format.
# The original CSV uses day/month/year and hour:minute.


datetime_columns = [
    "appointment_datetime",
    "booked_datetime"
]

for column in datetime_columns:

    if not pd.api.types.is_datetime64_any_dtype(df_clean[column]):

        df_clean[column] = pd.to_datetime(
            df_clean[column],
            format="%d/%m/%Y %H:%M",
            errors="coerce"
        )

# 3. Investigate datetime completeness and date ranges.

print("\n2. DATETIME VALIDATION")

for column in datetime_columns:

    missing_count = int(df_clean[column].isna().sum())

    print(f"\nColumn: {column}")
    print(f"Missing or invalid dates: {missing_count:,}")
    print(f"Earliest date: {df_clean[column].min()}")
    print(f"Latest date: {df_clean[column].max()}")

# 4. Calculate booking lead time.
# Booking lead time is the difference between the scheduled
# appointment timestamp and the original booking timestamp.

df_clean["booking_lead_days"] = (
    (
        df_clean["appointment_datetime"]
        - df_clean["booked_datetime"]
    ).dt.total_seconds()
    / 86400
)

print("\n3. BOOKING LEAD-TIME DISTRIBUTION")

display(
    df_clean["booking_lead_days"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

# 5. Identify missing or invalid datetime records.
# A missing appointment or booking timestamp prevents
# reliable calculation of booking lead time.

missing_datetime_mask = (
    df_clean[datetime_columns].isna().any(axis=1)
)

missing_datetime_count = int(
    missing_datetime_mask.sum()
)

# 6. Identify invalid booking chronology.
# A negative booking lead time means the recorded booking
# occurred after the scheduled appointment time.

invalid_chronology_mask = (
    df_clean["booking_lead_days"] < 0
)

invalid_chronology_count = int(
    invalid_chronology_mask.sum()
)

print("\n4. DATETIME AND CHRONOLOGY QUALITY CHECKS")

print(
    f"Missing or invalid datetime records: "
    f"{missing_datetime_count:,}"
)

print(
    f"Appointments booked after scheduled time: "
    f"{invalid_chronology_count:,}"
)

# 7. Examine invalid chronology by attendance status.


print("\n5. INVALID CHRONOLOGY BY ATTENDANCE STATUS")

display(
    df_clean.loc[
        invalid_chronology_mask,
        "attendance_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("Attendance_Status")
    .reset_index(name="Appointments")
)

# 8. Identify bookings with valid, nonnegative lead times.

valid_lead_mask = (
    df_clean["booking_lead_days"].notna()
    & (df_clean["booking_lead_days"] >= 0)
)

# 9. Identify same-calendar-day bookings.
# Same-day bookings are based on calendar dates.

same_day_mask = (
    valid_lead_mask
    & (
        df_clean["booked_datetime"].dt.normalize()
        == df_clean["appointment_datetime"].dt.normalize()
    )
)

# 10. Identify short-notice bookings.
# These are appointments booked less than seven days
# before the scheduled appointment.


short_notice_mask = (
    valid_lead_mask
    & (df_clean["booking_lead_days"] < 7)
)

# 11. Identify bookings made at least seven days ahead.

seven_days_or_more_mask = (
    valid_lead_mask
    & (df_clean["booking_lead_days"] >= 7)
)

print("\n6. BOOKING LEAD-TIME GROUPS")

print(
    f"Same-calendar-day bookings: "
    f"{int(same_day_mask.sum()):,}"
)

print(
    f"Bookings less than 7 days ahead: "
    f"{int(short_notice_mask.sum()):,}"
)

print(
    f"Bookings at least 7 days ahead: "
    f"{int(seven_days_or_more_mask.sum()):,}"
)

# 12. Investigate short-notice appointments by outcome.

print("\n7. SHORT-NOTICE APPOINTMENTS BY OUTCOME")

display(
    df_clean.loc[
        short_notice_mask,
        "attendance_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("Attendance_Status")
    .reset_index(name="Appointments")
)

# 13. Apply the agreed booking lead-time boundaries.
# The lower boundary is 0 days.
# The upper boundary is 301 days.
# This includes the five appointments whose original
# booking lead time was slightly above 300 days.

MIN_BOOKING_LEAD_DAYS = 0
MAX_BOOKING_LEAD_DAYS = 301

within_301_days_mask = (
    df_clean["booking_lead_days"].notna()
    & (
        df_clean["booking_lead_days"]
        >= MIN_BOOKING_LEAD_DAYS
    )
    & (
        df_clean["booking_lead_days"]
        <= MAX_BOOKING_LEAD_DAYS
    )
)

over_301_days_mask = (
    df_clean["booking_lead_days"]
    > MAX_BOOKING_LEAD_DAYS
)

print("\n8. AGREED 0–301-DAY BOOKING RANGE")

print(
    f"Valid bookings within 0–301 days: "
    f"{int(within_301_days_mask.sum()):,}"
)

print(
    f"Valid bookings above 301 days: "
    f"{int(over_301_days_mask.sum()):,}"
)

print(
    f"Invalid booking chronology: "
    f"{invalid_chronology_count:,}"
)

print(
    f"Missing or invalid datetime records: "
    f"{missing_datetime_count:,}"
)

# 14. Reconcile the booking lead-time categories.
# Every appointment must belong to exactly one category:
# valid within 301 days, above 301 days, negative lead time,
# or missing/unparseable datetime.

booking_category_total = (
    int(within_301_days_mask.sum())
    + int(over_301_days_mask.sum())
    + invalid_chronology_count
    + missing_datetime_count
)

assert booking_category_total == rows_before, (
    "Booking lead-time categories do not reconcile "
    "with the cleaned dataset."
)

print("\n9. BOOKING CATEGORY RECONCILIATION")

print(f"Total categorized appointments: {booking_category_total:,}")
print(f"Expected appointments: {rows_before:,}")
print("Reconciliation status: PASSED")

# 15. Define the dynamic prediction timing strategy.
# For appointments booked at least seven days in advance,
# the prediction is made seven days before the appointment.
# For short-notice bookings, the prediction is made at booking.
# This prevents excluding valid appointments booked
# within seven days of their scheduled date.

prediction_cutoff = pd.Series(
    pd.NaT,
    index=df_clean.index,
    dtype="datetime64[ns]"
)

prediction_cutoff.loc[seven_days_or_more_mask] = (
    df_clean.loc[
        seven_days_or_more_mask,
        "appointment_datetime"
    ] - pd.Timedelta(days=7)
)

prediction_cutoff.loc[short_notice_mask] = (
    df_clean.loc[
        short_notice_mask,
        "booked_datetime"
    ]
)

df_clean["prediction_cutoff_datetime"] = prediction_cutoff

# 16. Calculate time remaining until the appointment
# at the point when the prediction is made.

df_clean["days_until_appointment_at_prediction"] = (
    (
        df_clean["appointment_datetime"]
        - df_clean["prediction_cutoff_datetime"]
    ).dt.total_seconds()
    / 86400
)

# 17. Validate prediction timing.
# Predictions must not occur before the recorded booking
# or after the scheduled appointment.

prediction_timing_valid_mask = (
    df_clean["prediction_cutoff_datetime"].notna()
    & (
        df_clean["prediction_cutoff_datetime"]
        >= df_clean["booked_datetime"]
    )
    & (
        df_clean["prediction_cutoff_datetime"]
        <= df_clean["appointment_datetime"]
    )
)

print("\n10. DYNAMIC PREDICTION TIMING")

print(
    "Prediction timing for bookings at least 7 days ahead: "
    "7 days before appointment."
)

print(
    "Prediction timing for bookings less than 7 days ahead: "
    "at the time of booking."
)

print(
    f"Appointments with valid prediction timing: "
    f"{int(prediction_timing_valid_mask.sum()):,}"
)

print(
    f"Appointments within the agreed booking range: "
    f"{int(within_301_days_mask.sum()):,}"
)

# 18. Validate prediction timing for every appointment
# within the agreed booking lead-time range.

assert prediction_timing_valid_mask.loc[
    within_301_days_mask
].all(), (
    "One or more valid bookings have invalid prediction timing."
)

# 19. Confirm that the seven-day rule is applied correctly
# to appointments booked at least seven days ahead.

long_notice_prediction_days = (
    df_clean.loc[
        seven_days_or_more_mask,
        "days_until_appointment_at_prediction"
    ]
)

assert np.allclose(
    long_notice_prediction_days.dropna(),
    7.0
), (
    "Long-notice appointments do not have a seven-day "
    "prediction horizon."
)

# 20. Confirm that short-notice predictions are made
# at the original booking timestamp.

assert (
    df_clean.loc[
        short_notice_mask,
        "prediction_cutoff_datetime"
    ]
    == df_clean.loc[
        short_notice_mask,
        "booked_datetime"
    ]
).all(), (
    "Short-notice predictions are not aligned "
    "with booking time."
)

print("\n11. PREDICTION TIMING VALIDATION")
print("Long-notice prediction timing: PASSED")
print("Short-notice prediction timing: PASSED")
print("Prediction chronology: PASSED")

# 21. Save records with invalid datetime information
# or invalid booking chronology for audit purposes.

invalid_booking_records = df_clean.loc[
    missing_datetime_mask | invalid_chronology_mask
].copy()

invalid_booking_report_path = (
    CLEANING_REPORT_DIR
    / "invalid_booking_chronology.csv"
)

invalid_booking_records.to_csv(
    invalid_booking_report_path,
    index=False
)

print("\n12. INVALID BOOKING AUDIT REPORT")

print(f"Audit records: {len(invalid_booking_records):,}")
print(f"Report saved to: {invalid_booking_report_path}")

# 22. Record the cleaning and modelling decisions.

print("\n13. CLEANING AND MODELLING DECISIONS")

print(
    "Decision 1: Retain all appointments in the "
    "master cleaned dataset."
)

print(
    "Decision 2: Use booking lead times from "
    "0 to 301 days for modelling eligibility."
)

print(
    "Decision 3: Include valid same-day and "
    "short-notice appointments."
)

print(
    "Decision 4: Exclude invalid booking chronology "
    "from the modelling cohort, not the master dataset."
)

print(
    "Decision 5: Exclude cancellations from the "
    "binary Attended-versus-DNA modelling cohort."
)

print(
    "Decision 6: Use only information available "
    "at or before the prediction cutoff."
)

# 23. Final dataset validation.

assert len(df_clean) == rows_before == 40000, (
    "Unexpected change in appointment count."
)

assert df_clean["appointment_id"].is_unique, (
    "Duplicate appointment IDs detected."
)

assert len(df_raw) == 40180, (
    "Original dataset has changed."
)

assert df_clean["booking_lead_days"].notna().sum() == (
    rows_before - missing_datetime_count
), (
    "Unexpected missing booking lead-time values."
)

# 24. Final confirmation.

print("\nCELL 8 COMPLETED SUCCESSFULLY")
print("Datetime fields validated.")
print("Booking lead times calculated.")
print("Agreed 0–301-day range applied.")
print("Short-notice bookings retained.")
print("Dynamic prediction timestamps created.")
print("Invalid chronology audited.")
print("All 40,000 appointments preserved.")

DATETIME VALIDATION AND BOOKING LEAD-TIME ANALYSIS

1. DATASET INFORMATION
Total appointments: 40,000
Total columns before processing: 20

2. DATETIME VALIDATION

Column: appointment_datetime
Missing or invalid dates: 0
Earliest date: 2025-04-01 08:30:00
Latest date: 2026-03-31 16:30:00

Column: booked_datetime
Missing or invalid dates: 0
Earliest date: 2024-06-21 17:47:00
Latest date: 2026-04-13 09:00:00

3. BOOKING LEAD-TIME DISTRIBUTION


count    40000.000000
mean        45.806378
std         37.550361
min        -19.000000
1%           1.191646
5%           4.979063
25%         18.224132
50%         36.075000
75%         62.922396
95%        119.225764
99%        173.786410
max        300.275000
Name: booking_lead_days, dtype: float64


4. DATETIME AND CHRONOLOGY QUALITY CHECKS
Missing or invalid datetime records: 0
Appointments booked after scheduled time: 110

5. INVALID CHRONOLOGY BY ATTENDANCE STATUS


,Attendance_Status,Appointments
0,Attended,90
1,DNA,10
2,Cancelled by patient,9
3,Cancelled by hospital,1



6. BOOKING LEAD-TIME GROUPS
Same-calendar-day bookings: 50
Bookings less than 7 days ahead: 2,972
Bookings at least 7 days ahead: 36,918

7. SHORT-NOTICE APPOINTMENTS BY OUTCOME


,Attendance_Status,Appointments
0,Attended,2502
1,DNA,195
2,Cancelled by patient,179
3,Cancelled by hospital,96



8. AGREED 0–301-DAY BOOKING RANGE
Valid bookings within 0–301 days: 39,890
Valid bookings above 301 days: 0
Invalid booking chronology: 110
Missing or invalid datetime records: 0

9. BOOKING CATEGORY RECONCILIATION
Total categorized appointments: 40,000
Expected appointments: 40,000
Reconciliation status: PASSED

10. DYNAMIC PREDICTION TIMING
Prediction timing for bookings at least 7 days ahead: 7 days before appointment.
Prediction timing for bookings less than 7 days ahead: at the time of booking.
Appointments with valid prediction timing: 39,890
Appointments within the agreed booking range: 39,890

11. PREDICTION TIMING VALIDATION
Long-notice prediction timing: PASSED
Short-notice prediction timing: PASSED
Prediction chronology: PASSED

12. INVALID BOOKING AUDIT REPORT
Audit records: 110
Report saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\invalid_booking_chronology.csv

13. CLEANING AND MODELLING DECISIONS
Decision 1: Retain all appointme

In [21]:
# DEFINE THE BINARY DNA PREDICTION COHORT


# Objectives:
# 1. Preserve the master cleaned dataset.
# 2. Select Attended and DNA appointments.
# 3. Apply the agreed 0–301-day booking lead-time range.
# 4. Retain valid short-notice appointments.
# 5. Exclude invalid booking chronology from modelling.
# 6. Create the binary target variable.
# 7. Reconcile all included and excluded appointments.
# 8. Save an eligibility audit report.

print("CELL 9: DEFINE THE BINARY DNA PREDICTION COHORT")

# 1. Confirm the master dataset size.

master_count = len(df_clean)

assert master_count == 40000, (
    "Unexpected master dataset size."
)

print("\n1. MASTER DATASET")
print(f"Total cleaned appointments: {master_count:,}")

# 2. Define the two outcomes for binary prediction.
# DNA is the positive class.
# Cancellations are excluded from this binary task.

binary_outcomes = ["Attended", "DNA"]

binary_outcome_mask = (
    df_clean["attendance_status"].isin(binary_outcomes)
)

# 3. Define valid booking chronology and lead time.
# Use the 0–301-day range agreed in Cell 8.

valid_booking_mask = (
    df_clean["booking_lead_days"].between(
        MIN_BOOKING_LEAD_DAYS,
        MAX_BOOKING_LEAD_DAYS,
        inclusive="both"
    )
)

# 4. Confirm the prediction timestamp is valid.
# A prediction must be made at or after booking
# and no later than the scheduled appointment.

valid_prediction_mask = (
    df_clean["prediction_cutoff_datetime"].notna()
    & (
        df_clean["prediction_cutoff_datetime"]
        >= df_clean["booked_datetime"]
    )
    & (
        df_clean["prediction_cutoff_datetime"]
        <= df_clean["appointment_datetime"]
    )
)

# 5. Define final eligibility.
# All three conditions must be satisfied.

eligible_mask = (
    binary_outcome_mask
    & valid_booking_mask
    & valid_prediction_mask
)

# 6. Create the modelling dataset.
# This is a separate copy.
# The master dataset remains unchanged.

df_model = df_clean.loc[eligible_mask].copy()

df_model = df_model.reset_index(drop=True)

# 7. Create the binary target.
# 1 = Did Not Attend (DNA)
# 0 = Attended

df_model["dna_target"] = (
    df_model["attendance_status"]
    .map({
        "Attended": 0,
        "DNA": 1
    })
    .astype(int)
)

print("\n2. BINARY MODELLING COHORT")

print(f"Eligible appointments: {len(df_model):,}")

display(
    df_model["attendance_status"]
    .value_counts()
    .rename_axis("Attendance_Status")
    .reset_index(name="Appointments")
)

# 8. Calculate the DNA rate.

dna_count = int(df_model["dna_target"].sum())

attended_count = int(
    (df_model["dna_target"] == 0).sum()
)

dna_rate = (
    dna_count / len(df_model) * 100
    if len(df_model) > 0 else 0
)

print("\n3. TARGET DISTRIBUTION")

print(f"Attended appointments: {attended_count:,}")
print(f"DNA appointments: {dna_count:,}")
print(f"DNA rate: {dna_rate:.2f}%")

# 9. Investigate the booking lead-time distribution
# of the eligible modelling population.

print("\n4. BOOKING LEAD TIME IN MODELLING COHORT")

display(
    df_model["booking_lead_days"].describe()
)

# 10. Confirm short-notice appointments are retained.

model_short_notice_mask = (
    df_model["booking_lead_days"] < 7
)

model_long_notice_mask = (
    df_model["booking_lead_days"] >= 7
)

print("\n5. PREDICTION TIMING GROUPS")

print(
    f"Eligible short-notice appointments: "
    f"{int(model_short_notice_mask.sum()):,}"
)

print(
    f"Eligible appointments booked 7+ days ahead: "
    f"{int(model_long_notice_mask.sum()):,}"
)

# 11. Reconcile excluded appointments.
# Give each record one exclusion reason.
# This prevents double counting.

eligibility_audit = df_clean[
    [
        "appointment_id",
        "attendance_status",
        "booking_lead_days",
        "prediction_cutoff_datetime"
    ]
].copy()

eligibility_audit["eligible_for_model"] = eligible_mask

eligibility_audit["exclusion_reason"] = np.select(
    [
        ~binary_outcome_mask,
        binary_outcome_mask & ~valid_booking_mask,
        binary_outcome_mask
        & valid_booking_mask
        & ~valid_prediction_mask
    ],
    [
        "Cancellation or other non-binary outcome",
        "Invalid or out-of-range booking lead time",
        "Invalid prediction timestamp"
    ],
    default="Eligible"
)

print("\n6. ELIGIBILITY RECONCILIATION")

display(
    eligibility_audit["exclusion_reason"]
    .value_counts()
    .rename_axis("Eligibility_Status")
    .reset_index(name="Appointments")
)

excluded_count = int((~eligible_mask).sum())

print(f"\nMaster appointments: {master_count:,}")
print(f"Eligible appointments: {len(df_model):,}")
print(f"Excluded appointments: {excluded_count:,}")

assert len(df_model) + excluded_count == master_count, (
    "Eligibility reconciliation failed."
)

# 12. Validate the final modelling population.

assert df_model["appointment_id"].is_unique, (
    "Duplicate appointment IDs in modelling dataset."
)

assert df_model["dna_target"].isin([0, 1]).all(), (
    "Invalid binary target values."
)

assert df_model["booking_lead_days"].between(
    0, 301
).all(), (
    "Invalid booking lead time in modelling dataset."
)

assert (
    df_model["prediction_cutoff_datetime"]
    >= df_model["booked_datetime"]
).all(), (
    "Prediction occurs before booking."
)

assert (
    df_model["prediction_cutoff_datetime"]
    <= df_model["appointment_datetime"]
).all(), (
    "Prediction occurs after appointment."
)

assert len(df_clean) == 40000, (
    "Master dataset was unexpectedly modified."
)

# 13. Save the eligibility audit report.

eligibility_report_path = (
    CLEANING_REPORT_DIR
    / "modelling_eligibility_audit.csv"
)

eligibility_audit.to_csv(
    eligibility_report_path,
    index=False
)

print("\n7. ELIGIBILITY AUDIT REPORT")
print(f"Report saved to: {eligibility_report_path}")

# 14. Document the modelling decision.

print("\n8. MODELLING DECISION")

print("Target: DNA versus Attended")
print("DNA = 1")
print("Attended = 0")
print("Booking lead-time range: 0–301 days")
print("Short-notice appointments: Included")
print("Cancellations: Excluded")
print("Invalid booking chronology: Excluded")

print(
    "Historical predictors must still be checked "
    "for availability at the prediction cutoff."
)

print("\nCELL 9 COMPLETED SUCCESSFULLY")
print("Binary modelling cohort created.")
print("Dynamic prediction strategy retained.")
print("Eligibility reconciliation passed.")
print("Master cleaned dataset preserved.")

CELL 9: DEFINE THE BINARY DNA PREDICTION COHORT

1. MASTER DATASET
Total cleaned appointments: 40,000

2. BINARY MODELLING COHORT
Eligible appointments: 36,264


,Attendance_Status,Appointments
0,Attended,32958
1,DNA,3306



3. TARGET DISTRIBUTION
Attended appointments: 32,958
DNA appointments: 3,306
DNA rate: 9.12%

4. BOOKING LEAD TIME IN MODELLING COHORT


count    36264.000000
mean        45.999920
std         37.499306
min          0.020833
25%         18.711979
50%         36.168056
75%         62.993229
max        300.275000
Name: booking_lead_days, dtype: float64


5. PREDICTION TIMING GROUPS
Eligible short-notice appointments: 2,697
Eligible appointments booked 7+ days ahead: 33,567

6. ELIGIBILITY RECONCILIATION


,Eligibility_Status,Appointments
0,Eligible,36264
1,Cancellation or other non-binary outcome,3636
2,Invalid or out-of-range booking lead time,100



Master appointments: 40,000
Eligible appointments: 36,264
Excluded appointments: 3,736

7. ELIGIBILITY AUDIT REPORT
Report saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\modelling_eligibility_audit.csv

8. MODELLING DECISION
Target: DNA versus Attended
DNA = 1
Attended = 0
Booking lead-time range: 0–301 days
Short-notice appointments: Included
Cancellations: Excluded
Invalid booking chronology: Excluded
Historical predictors must still be checked for availability at the prediction cutoff.

CELL 9 COMPLETED SUCCESSFULLY
Binary modelling cohort created.
Dynamic prediction strategy retained.
Eligibility reconciliation passed.
Master cleaned dataset preserved.


In [22]:
# VALIDATE THE FINAL MODELLING COHORT

# Objectives:
# 1. Validate the master and modelling dataset sizes.
# 2. Check duplicate and missing identifiers.
# 3. Verify the binary target distribution.
# 4. Validate booking lead times and prediction timestamps.
# 5. Confirm short-notice appointments are retained.
# 6. Review missing predictor values.
# 7. Identify columns that must not enter the model.
# 8. Confirm readiness for EDA and feature engineering.

print("CELL 10: FINAL MODELLING COHORT VALIDATION")

# 1. Check dataset sizes.

print("\n1. DATASET DIMENSIONS")

print(f"Original records: {len(df_raw):,}")
print(f"Master cleaned records: {len(df_clean):,}")
print(f"Eligible modelling records: {len(df_model):,}")

assert len(df_raw) == 40180
assert len(df_clean) == 40000
assert len(df_model) == 36264

# 2. Validate identifiers.
# Each appointment must appear only once.
# A patient may legitimately have multiple appointments.

print("\n2. IDENTIFIER VALIDATION")

for name, dataset in [
    ("Master", df_clean),
    ("Modelling", df_model)
]:
    duplicate_ids = int(
        dataset["appointment_id"].duplicated().sum()
    )

    missing_ids = int(
        dataset["appointment_id"].isna().sum()
    )

    print(f"\n{name} dataset:")
    print(f"Duplicate appointment IDs: {duplicate_ids:,}")
    print(f"Missing appointment IDs: {missing_ids:,}")

    assert duplicate_ids == 0
    assert missing_ids == 0

# 3. Validate the binary target.

print("\n3. TARGET VALIDATION")

target_summary = (
    df_model["dna_target"]
    .value_counts()
    .sort_index()
)

print(f"Attended (0): {target_summary.get(0, 0):,}")
print(f"DNA (1): {target_summary.get(1, 0):,}")

dna_rate = df_model["dna_target"].mean() * 100

print(f"DNA rate: {dna_rate:.2f}%")

assert df_model["dna_target"].isin([0, 1]).all()

assert (
    df_model.loc[
        df_model["attendance_status"] == "DNA",
        "dna_target"
    ] == 1
).all()

assert (
    df_model.loc[
        df_model["attendance_status"] == "Attended",
        "dna_target"
    ] == 0
).all()

# 4. Validate booking lead times.

print("\n4. BOOKING LEAD-TIME VALIDATION")

minimum_lead = df_model["booking_lead_days"].min()
maximum_lead = df_model["booking_lead_days"].max()

print(f"Minimum lead time: {minimum_lead:.3f} days")
print(f"Maximum lead time: {maximum_lead:.3f} days")

assert df_model["booking_lead_days"].between(
    0, 301
).all()

# 5. Validate prediction timestamps.

print("\n5. PREDICTION TIMESTAMP VALIDATION")

assert df_model["prediction_cutoff_datetime"].notna().all()

assert (
    df_model["prediction_cutoff_datetime"]
    >= df_model["booked_datetime"]
).all()

assert (
    df_model["prediction_cutoff_datetime"]
    <= df_model["appointment_datetime"]
).all()

print("All prediction timestamps are chronologically valid.")

# 6. Confirm short-notice appointments are included.

print("\n6. SHORT-NOTICE APPOINTMENT VALIDATION")

short_notice = (
    df_model["booking_lead_days"] < 7
)

long_notice = (
    df_model["booking_lead_days"] >= 7
)

short_notice_count = int(short_notice.sum())
long_notice_count = int(long_notice.sum())

print(f"Short-notice appointments: {short_notice_count:,}")
print(f"Appointments booked 7+ days ahead: {long_notice_count:,}")

assert short_notice_count + long_notice_count == len(df_model)

# Check prediction at booking for short-notice appointments.

assert (
    df_model.loc[
        short_notice,
        "prediction_cutoff_datetime"
    ]
    == df_model.loc[
        short_notice,
        "booked_datetime"
    ]
).all()

# Check prediction seven days before appointment
# for appointments booked at least seven days ahead.

assert np.allclose(
    df_model.loc[
        long_notice,
        "days_until_appointment_at_prediction"
    ],
    7.0
)

print("Dynamic prediction timing validated.")

# 7. Review missing values in potential predictors.
# Missing values will be handled during model training.


print("\n7. POTENTIAL PREDICTOR MISSING VALUES")

candidate_predictors = [
    "age",
    "sex",
    "imd_decile",
    "ethnic_group",
    "distance_km",
    "interpreter_required",
    "long_term_conditions",
    "specialty",
    "appointment_type",
    "consultation_mode",
    "referral_source",
    "booking_lead_days"
]

missing_predictors = pd.DataFrame({
    "feature": candidate_predictors,
    "missing_count": [
        int(df_model[column].isna().sum())
        for column in candidate_predictors
    ],
    "missing_percentage": [
        round(df_model[column].isna().mean() * 100, 2)
        for column in candidate_predictors
    ]
})

display(missing_predictors)

# 8. Identify columns requiring exclusion or
# additional point-in-time validation.

print("\n8. FEATURE LEAKAGE REVIEW")

excluded_features = [
    "appointment_id",
    "patient_id",
    "attendance_status",
    "dna_target",
    "clinic_checkin_time"
]

historical_features_to_audit = [
    "previous_dnas_12m",
    "rescheduled_count",
    "sms_reminder_sent"
]

print("\nIdentifiers, outcome and leakage columns:")
for column in excluded_features:
    print(f"- {column}")

print("\nHistorical features requiring availability checks:")
for column in historical_features_to_audit:
    print(f"- {column}")

print(
    "\nThese historical fields must not be used until "
    "we can establish that their values were known "
    "at the relevant prediction timestamp."
)

# 9. Confirm that the modelling dataset contains
# only eligible appointment outcomes.

print("\n9. FINAL ELIGIBILITY CHECK")

assert set(
    df_model["attendance_status"].unique()
) == {"Attended", "DNA"}

assert df_model["appointment_id"].isin(
    df_clean["appointment_id"]
).all()

print("All modelling appointments originate from the master dataset.")

# 10. Final summary.

print("\n10. VALIDATION SUMMARY")

print("Master dataset: PASSED")
print("Appointment identifiers: PASSED")
print("Binary target: PASSED")
print("Booking lead-time range: PASSED")
print("Dynamic prediction timestamps: PASSED")
print("Short-notice appointment retention: PASSED")
print("Eligibility: PASSED")

print("\nCELL 10 COMPLETED SUCCESSFULLY")
print("Final modelling cohort validated.")
print("No records removed or modified.")
print("Ready for dataset export and EDA updates.")

CELL 10: FINAL MODELLING COHORT VALIDATION

1. DATASET DIMENSIONS
Original records: 40,180
Master cleaned records: 40,000
Eligible modelling records: 36,264

2. IDENTIFIER VALIDATION

Master dataset:
Duplicate appointment IDs: 0
Missing appointment IDs: 0

Modelling dataset:
Duplicate appointment IDs: 0
Missing appointment IDs: 0

3. TARGET VALIDATION
Attended (0): 32,958
DNA (1): 3,306
DNA rate: 9.12%

4. BOOKING LEAD-TIME VALIDATION
Minimum lead time: 0.021 days
Maximum lead time: 300.275 days

5. PREDICTION TIMESTAMP VALIDATION
All prediction timestamps are chronologically valid.

6. SHORT-NOTICE APPOINTMENT VALIDATION
Short-notice appointments: 2,697
Appointments booked 7+ days ahead: 33,567
Dynamic prediction timing validated.

7. POTENTIAL PREDICTOR MISSING VALUES


,feature,missing_count,missing_percentage
0,age,48,0.13
1,sex,0,0.00
2,imd_decile,1265,3.49
3,ethnic_group,0,0.00
4,distance_km,362,1.00
5,interpreter_required,0,0.00
6,long_term_conditions,0,0.00
7,specialty,0,0.00
8,appointment_type,0,0.00
9,consultation_mode,0,0.00



8. FEATURE LEAKAGE REVIEW

Identifiers, outcome and leakage columns:
- appointment_id
- patient_id
- attendance_status
- dna_target
- clinic_checkin_time

Historical features requiring availability checks:
- previous_dnas_12m
- rescheduled_count
- sms_reminder_sent

These historical fields must not be used until we can establish that their values were known at the relevant prediction timestamp.

9. FINAL ELIGIBILITY CHECK
All modelling appointments originate from the master dataset.

10. VALIDATION SUMMARY
Master dataset: PASSED
Appointment identifiers: PASSED
Binary target: PASSED
Booking lead-time range: PASSED
Dynamic prediction timestamps: PASSED
Short-notice appointment retention: PASSED
Eligibility: PASSED

CELL 10 COMPLETED SUCCESSFULLY
Final modelling cohort validated.
No records removed or modified.
Ready for dataset export and EDA updates.


In [23]:
# CELL 11: SAVE CLEANED MASTER AND MODELLING DATASETS


# Objectives:
# 1. Create the processed-data directory.
# 2. Save the cleaned master dataset.
# 3. Save the eligible binary modelling dataset.
# 4. Save a dataset summary for auditing.
# 5. Reload the exported files to verify their integrity.
# 6. Confirm that the original raw dataset remains unchanged.

print("CELL 11: SAVE CLEANED DATASETS")

# 1. Define the processed-data directory.

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\n1. OUTPUT DIRECTORY")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

# 2. Define output file paths.
# These names distinguish the complete cleaned dataset
# from the binary Attended-versus-DNA modelling cohort.

master_output_path = (
    PROCESSED_DATA_DIR / "outpatient_appointments_cleaned.csv"
)

model_output_path = (
    PROCESSED_DATA_DIR / "outpatient_dna_modelling.csv"
)

summary_output_path = (
    CLEANING_REPORT_DIR / "final_dataset_summary.csv"
)

# 3. Validate the datasets before exporting.

assert len(df_clean) == 40000, (
    "Unexpected master dataset size."
)

assert len(df_model) == 36264, (
    "Unexpected modelling dataset size."
)

assert df_clean["appointment_id"].is_unique, (
    "Duplicate appointment IDs in master dataset."
)

assert df_model["appointment_id"].is_unique, (
    "Duplicate appointment IDs in modelling dataset."
)

assert df_model["dna_target"].isin([0, 1]).all(), (
    "Invalid binary target values."
)

assert df_model["booking_lead_days"].between(
    0, 301
).all(), (
    "Invalid booking lead times in modelling dataset."
)

# 4. Save the cleaned master dataset.
# It retains attended, DNA, and cancelled appointments,
# including records excluded from binary modelling.

df_clean.to_csv(
    master_output_path,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("\n2. MASTER DATASET EXPORTED")
print(f"File: {master_output_path}")
print(f"Rows: {len(df_clean):,}")
print(f"Columns: {df_clean.shape[1]}")

# 5. Save the binary modelling dataset.
# It includes eligible Attended and DNA appointments,
# their prediction timestamps, and the binary target.


df_model.to_csv(
    model_output_path,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("\n3. MODELLING DATASET EXPORTED")
print(f"File: {model_output_path}")
print(f"Rows: {len(df_model):,}")
print(f"Columns: {df_model.shape[1]}")

# 6. Create a final dataset summary.

final_dataset_summary = pd.DataFrame([
    {
        "dataset": "Original raw dataset",
        "appointments": len(df_raw),
        "description": "Unmodified source dataset"
    },
    {
        "dataset": "Cleaned master dataset",
        "appointments": len(df_clean),
        "description": "All unique appointments after cleaning"
    },
    {
        "dataset": "Binary modelling dataset",
        "appointments": len(df_model),
        "description": (
            "Eligible Attended and DNA appointments "
            "with valid booking and prediction timestamps"
        )
    },
    {
        "dataset": "Modelling Attended appointments",
        "appointments": int(
            (df_model["dna_target"] == 0).sum()
        ),
        "description": "Negative class"
    },
    {
        "dataset": "Modelling DNA appointments",
        "appointments": int(
            (df_model["dna_target"] == 1).sum()
        ),
        "description": "Positive class"
    },
    {
        "dataset": "Short-notice modelling appointments",
        "appointments": int(
            (df_model["booking_lead_days"] < 7).sum()
        ),
        "description": "Predicted at booking"
    }
])

final_dataset_summary.to_csv(
    summary_output_path,
    index=False
)

print("\n4. FINAL DATASET SUMMARY")
display(final_dataset_summary)

print(f"\nSummary saved to: {summary_output_path}")

# 7. Reload both exported datasets.
# This verifies that the files can be read by later notebooks.

master_reloaded = pd.read_csv(
    master_output_path
)

model_reloaded = pd.read_csv(
    model_output_path
)

print("\n5. EXPORT VERIFICATION")

print(
    f"Reloaded master appointments: "
    f"{len(master_reloaded):,}"
)

print(
    f"Reloaded modelling appointments: "
    f"{len(model_reloaded):,}"
)

# 8. Validate the exported datasets.

assert master_reloaded.shape == df_clean.shape, (
    "Master export dimensions do not match."
)

assert model_reloaded.shape == df_model.shape, (
    "Modelling export dimensions do not match."
)

assert master_reloaded["appointment_id"].is_unique, (
    "Duplicate IDs detected in master export."
)

assert model_reloaded["appointment_id"].is_unique, (
    "Duplicate IDs detected in modelling export."
)

assert int(model_reloaded["dna_target"].sum()) == 3306, (
    "Exported DNA count does not match."
)

assert int(
    (model_reloaded["booking_lead_days"] < 7).sum()
) == 2697, (
    "Short-notice appointments were not fully retained."
)

# 9. Confirm that all original appointments remain
# available in the raw dataset.

assert len(df_raw) == 40180, (
    "Original raw dataset has changed."
)

# 10. Document the final modelling assumptions.

print("\n6. FINAL MODELLING ASSUMPTIONS")

print("Target: DNA (1) versus Attended (0)")
print("Booking lead-time range: 0–301 days")
print("Bookings less than 7 days ahead: predict at booking")
print("Bookings 7+ days ahead: predict 7 days before appointment")
print("Cancellations: excluded from binary modelling")
print("Invalid booking chronology: excluded from modelling")

print(
    "Historical features will be audited for "
    "point-in-time availability before model training."
)

print("\nCELL 11 COMPLETED SUCCESSFULLY")
print("Cleaned master dataset saved.")
print("Binary modelling dataset saved.")
print("Final summary saved.")
print("Export validation passed.")
print("Original raw dataset preserved.")

CELL 11: SAVE CLEANED DATASETS

1. OUTPUT DIRECTORY
Processed data directory: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\data\processed

2. MASTER DATASET EXPORTED
File: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\data\processed\outpatient_appointments_cleaned.csv
Rows: 40,000
Columns: 23

3. MODELLING DATASET EXPORTED
File: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\data\processed\outpatient_dna_modelling.csv
Rows: 36,264
Columns: 24

4. FINAL DATASET SUMMARY


,dataset,appointments,description
0,Original raw dataset,40180,Unmodified source dataset
1,Cleaned master dataset,40000,All unique appointments after cleaning
2,Binary modelling dataset,36264,Eligible Attended and DNA appointments with va...
3,Modelling Attended appointments,32958,Negative class
4,Modelling DNA appointments,3306,Positive class
5,Short-notice modelling appointments,2697,Predicted at booking



Summary saved to: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\cleaning\final_dataset_summary.csv

5. EXPORT VERIFICATION
Reloaded master appointments: 40,000
Reloaded modelling appointments: 36,264

6. FINAL MODELLING ASSUMPTIONS
Target: DNA (1) versus Attended (0)
Booking lead-time range: 0–301 days
Bookings less than 7 days ahead: predict at booking
Bookings 7+ days ahead: predict 7 days before appointment
Cancellations: excluded from binary modelling
Invalid booking chronology: excluded from modelling
Historical features will be audited for point-in-time availability before model training.

CELL 11 COMPLETED SUCCESSFULLY
Cleaned master dataset saved.
Binary modelling dataset saved.
Final summary saved.
Export validation passed.
Original raw dataset preserved.


Cell 11 has completed successfully. I've checked the output in your uploaded notebook, and both datasets were exported and reloaded without errors.
Dataset	Rows	Columns
Original raw dataset	40,180	20
Cleaned master dataset	40,000	23
Binary modeling dataset	36,264	24

The modeling dataset contains 32,958 attended appointments, 3,306 DNAs, and 2,697 short-notice appointments. The final DNA rate is approximately 9.12%.

The files were saved to your project's data/processed folder, and the summary report was saved to reports/cleaning.
Next step: Review the remaining cleaning cells
The uploaded notebook currently contains completed Cells 1–11. Before we move to EDA, I recommend adding Cell 12: Final Cleaning Summary and Quality Assurance.

This will document our cleaning decisions, exclusions, final modeling population, and safeguards against data leakage in one place—useful for your NHS Dorset interview.

We'll continue with Cell 12 next, then update the EDA notebook using the new 36,264-row modeling dataset.

summary of the cleaned notebook
NHS Dorset DNA Prediction — Data Cleaning Summary
Notebook: 02_data_cleaning.ipynb

Purpose: To prepare a reliable, auditable dataset for predicting which outpatient appointments are at risk of being missed (Did Not Attend, or DNA), while minimizing data leakage and preserving valid appointment records.

1. Final dataset overview
Original records

40,180
Cleaned records

40,000
Modeling records

36,264
DNA rate

9.12%
Modeling cohort composition

Attended: 32,958 (90.88%)

DNA: 3,306 (9.12%)

2. Cleaning decisions and justifications
Data quality issue

Finding

Action and justification

Exact duplicates

180 records (0.45%)

Removed to prevent double-counting appointments

Invalid ages

54 values including −1, 121, 150 and 999

Replaced with NaN, retaining the appointments

Inconsistent sex categories

Seven original representations

Standardized into Female, Male and Unknown

Specialty spelling errors

Gastroentterology, Opthalmology

Corrected to Gastroenterology and Ophthalmology

Missing IMD decile

1,400 records

Retained; imputation deferred until modeling

Missing distance

400 records

Retained; imputation deferred until modeling

Missing check-in time

6,952 records

Retained in master data, but excluded as a predictor because of target leakage

Invalid booking chronology

110 records

Retained in master data, excluded from binary modeling

Canceled appointments

3,636 records

Excluded from the Attended-versus-DNA binary modeling task

Important: We removed only the 180 exact duplicate records from the master dataset. Other problematic records were corrected, flagged, or excluded from the separate modeling population rather than deleted from the cleaned master.

3. Modeling eligibility and exclusions
Reconciliation from cleaned master to modeling cohort

Cleaned master dataset

40,000

Canceled appointments

−3,636

Invalid chronology among Attended/DNA

−100

Final eligible appointments

36,264
The binary target is defined as dna_target = 1 for DNA and dna_target = 0 for Attended.

Cancellations are not treated as DNAs because they represent different outcomes and would change the meaning of the prediction problem.

4. Dynamic prediction strategy
We replaced the earlier fixed seven-day eligibility rule with a strategy that also supports short-notice appointments.

Original booking lead time

Prediction point

0 to less than 7 days

At booking

7 to 301 days

Seven days before appointment

The maximum observed valid booking lead time is approximately 300.275 days, so the 301-day upper limit retains these legitimate records.

This approach includes 2,697 eligible short-notice appointments that were excluded under our previous fixed seven-day rule.

5. Data leakage prevention
The notebook establishes several safeguards:

clinic_checkin_time must not be used for prediction because it reveals information associated with the attendance outcome.

previous_dnas_12m, rescheduled_count, and sms_reminder_sent require further checks to establish whether their values were known at the prediction cutoff.

Missing-value imputation will be fitted on training data only, rather than the complete dataset.

Appointment and patient identifiers will not be used directly as predictors.

Chronological splitting will be applied during modeling to evaluate performance on later appointments.

The prediction cutoff is calculated, but the point-in-time availability of every candidate feature has not yet been fully established.

6. Outputs saved
File

Purpose

data/processed/outpatient_appointments_cleaned.csv

Full cleaned dataset of 40,000 appointments

data/processed/outpatient_dna_modelling.csv

Eligible binary modeling dataset of 36,264 appointments

reports/cleaning/final_dataset_summary.csv

Dataset reconciliation and final counts

reports/cleaning/modelling_eligibility_audit.csv

Eligibility decisions and exclusion reasons

The notebook also saves reports for duplicates, invalid ages, categorical values, missing values, and booking chronology.

7. Overall conclusion
The cleaning notebook has produced a consistent and validated modeling dataset. It preserves valid appointments, documents exclusions, supports both standard and short-notice bookings, and establishes a clear prediction timestamp.

Interview-ready summary:

I started with 40,180 outpatient appointment records and removed 180 exact duplicates. I corrected invalid age values, standardized inconsistent categorical fields, investigated missing data, and validated booking chronology. Rather than unnecessarily deleting records, I preserved the cleaned master dataset and created a separate binary modeling population of 36,264 appointments, with a DNA rate of 9.12%. I also introduced a dynamic prediction strategy to include short-notice bookings and established safeguards against data leakage.

Next: We should update the EDA notebook to use the new 36,264-row modeling cohort, then proceed to feature engineering, model improvement, fairness assessment, and selection of the 150 patients for weekly outreach.